In [2]:
# Ao rodar pela primeira vez, execute o comando abaixo:

# %pip install -r requirements.txt

### Cenário A: O Pipeline Baseline (A Visão Tradicional do Hospital)
Objetivo: Estabelecer a nossa linha de base comparativa (Baseline) testando os algoritmos puros com as variáveis originais, priorizando a segurança clínica (alto Recall) para capturar o máximo de doentes na triagem.
- Bloco 1 (Importações): Prepara o ecossistema carregando as bibliotecas essenciais na memória. [Rodar 1º]
- Bloco 2 (Carga e Auditoria): Carrega a base Vigitel e executa a limpeza automática contra vazamento de dados (data leakage). [Rodar 2º]
- Bloco 3 (Split e Preprocessor): Garante a divisão estatística correta de treino/teste (20%) e configura o tratamento automatizado de nulos e escala. [Rodar 3º]
- Bloco 4 (Pipeline Original - Baseline): Roda a maratona de treinamento dos 3 modelos da equipe e gera todos os gráficos e relatórios comparativos para o PDF. [Rodar 4º]

### Cenário B: Otimização de Precisão (A Visão de Eficiência Médica)
Objetivo: Mitigar a "fadiga de alarmes" da equipe médica. Usamos as novas variáveis clínicas combinadas, removemos o peso artificial das classes e subimos o sarrafo do gatilho para 60%, garantindo que o modelo só dê o diagnóstico quando tiver alta certeza.
- Bloco 1 (Importações): Inicializa o ecossistema de código. [Rodar 1º]
- Bloco 2 (Carga e Auditoria): Prepara e audita os dados contra vazamentos. [Rodar 2º]
- Bloco 3 (Split e Preprocessor): Executa a quebra estatística das bases. [Rodar 3º]
- Bloco 3.5 (Engenharia de Variáveis): Injeta os 4 novos indicadores clínicos cruzados em cópias isoladas para dar mais poder preditivo ao modelo. [Rodar 4º]
- Bloco 4.5 (Otimização de Precisão): Treina a Random Forest customizada focando em assertividade absoluta. O limite de 60% faz a Precisão saltar para 74%, limpando os alarmes falsos. [Rodar 5º]

Nota: O Bloco 4 e o Bloco 4.6 são ignorados neste cenário para economizar tempo de processamento.

### Cenário C: Isolamento Científico (O Impacto Puro das Novas Variáveis)
Objetivo: Validar o real valor técnico das nossas colunas customizadas. Mantemos o modelo com as mesmíssimas regras de corte (50%) e pesos do grupo para responder matematicamente: as novas features, sozinhas, deixaram o modelo mais inteligente?
- Bloco 1 (Importações) $\rightarrow$ Bloco 2 (Carga) $\rightarrow$ Bloco 3 (Split): Preparação padrão da esteira de dados. [Rodar 1º, 2º e 3º]
- Bloco 3.5 (Engenharia de Variáveis): Acopla o Score de Síndrome Metabólica, Balanço Nutricional e os demais cruzamentos clínicos. [Rodar 4º]
- Bloco 4.6 (Apenas Novas Variáveis): Roda a Random Forest no limiar clássico de 50%. [Rodar 5º]
O veredito científico: O modelo escolhe caminhar mais fundo na árvore (max_depth: 16), provando que as novas variáveis entregaram padrões úteis que o modelo original não enxergava.

## Bloco 1: Importações e Funções de Sanitização de Texto
Organização de todas as dependências do ecossistema e algoritmos de tratamento textual para padronizar respostas de inquérito.

In [3]:
import os
from datetime import datetime
import numpy as np
import pandas as pd
import unicodedata
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve, average_precision_score, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def normalizar_texto(valor):
    if pd.isna(valor): return valor
    valor = str(valor).strip().lower()
    valor = unicodedata.normalize("NFKD", valor)
    return "".join([c for c in valor if not unicodedata.combining(c)])

def converter_sim_nao(valor):
    if pd.isna(valor): return valor
    valor_norm = normalizar_texto(valor)
    mapa = {"sim": 1, "s": 1, "yes": 1, "y": 1, "nao": 0, "não": 0, "n": 0, "no": 0}
    return mapa.get(valor_norm, valor)

## Bloco 2: Configurações de Diretório, Carga Otimizada e Data Leakage
Definição de caminhos dinâmicos, filtragem inicial baseada no dicionário de siglas (variaveis.txt) e expurgo automático de variáveis de vazamento que revelavam o diagnóstico médico antes da hora.

In [4]:
import re

PASTA_PROJETO = os.getcwd()
ARQUIVO_BASE_ORIGINAL = os.path.join(PASTA_PROJETO, "vigitel.csv")
ARQUIVO_VARIAVEIS = os.path.join(PASTA_PROJETO, "variaveis.txt")
ARQUIVO_BASE_TRATADA = os.path.join(PASTA_PROJETO, "base_tratada_sem_vazamento.csv")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PASTA_RESULTADOS = os.path.join(PASTA_PROJETO, "resultados_sem_vazamento", RUN_ID)
PASTA_MODELOS = os.path.join(PASTA_RESULTADOS, "modelos_salvos")
os.makedirs(PASTA_RESULTADOS, exist_ok=True)
os.makedirs(PASTA_MODELOS, exist_ok=True)

TARGET = "hart"
RANDOM_STATE = 42
N_FOLDS_AVALIACAO = 5
N_FOLDS_TUNING = 3
LIMITE_ALERTA_CORRELACAO = 0.30

# AJUSTE AQUI: cada um configura esse numero de acordo com os nucleos da
# propria maquina. Evite usar -1 (todos os nucleos) dentro do GridSearchCV
# e do cross_validate - foi exatamente isso, combinado com a Random Forest
# tambem pedindo todos os nucleos (paralelismo aninhado), que causou o
# "WinError 1450" no Windows.
N_JOBS_CONFIGURADO = 12

lista_variaveis = pd.read_csv(ARQUIVO_VARIAVEIS, header=None)[0].astype(str).str.strip().tolist()
df = pd.read_csv(ARQUIVO_BASE_ORIGINAL, encoding="latin1", sep=",", on_bad_lines="skip")

variaveis_validas = [col for col in lista_variaveis if col in df.columns]
variaveis_ausentes = [col for col in lista_variaveis if col not in df.columns]
df = df[variaveis_validas]

if variaveis_ausentes:
    print(f"Atencao: {len(variaveis_ausentes)} variavel(eis) do variaveis.txt NAO encontrada(s) na base:")
    for col in variaveis_ausentes:
        print("-", col)

colunas_texto = df.select_dtypes(include=["object", "string"]).columns
df[colunas_texto] = df[colunas_texto].apply(lambda col: col.map(converter_sim_nao))
df = df.dropna(subset=[TARGET])

y = df[TARGET]
X = df.drop([TARGET], axis=1)

# CORRECAO: antes, a checagem "termo in col.lower()" pegava "pressao" mesmo
# dentro de "depressao" (falso positivo), removendo essa variavel por engano
# e quebrando silenciosamente a feature "indice_estresse_sono" do Bloco 3.5.
# Agora so bloqueia o termo quando ele NAO esta colado depois de outra letra
# (ex.: "de" + "pressao"), mas continua pegando "hipertensao", "has_anos" etc.
termos_vazamento = ["has", "hipert", "hipertens", "pressao", "pressão"]


def contem_termo_vazamento(col_lower, termos):
    for termo in termos:
        if re.search(rf"(?<![a-z]){re.escape(termo)}", col_lower):
            return True
    return False


variaveis_remover_vazamento = [col for col in X.columns if contem_termo_vazamento(col.lower(), termos_vazamento)]
X = X.drop(columns=variaveis_remover_vazamento)

# Auditoria de vazamento residual por correlacao - nao remove nada
# automaticamente, so avisa para revisao manual.
colunas_numericas_auditoria = X.select_dtypes(include=["int64", "float64"]).columns
correlacoes_target = X[colunas_numericas_auditoria].corrwith(y).abs().sort_values(ascending=False)
suspeitas_correlacao = correlacoes_target[correlacoes_target > LIMITE_ALERTA_CORRELACAO]

print(f"\nBase carregada. Linhas: {X.shape[0]} | Colunas restantes: {X.shape[1]}")
print("Variaveis removidas por possivel vazamento (nome):", variaveis_remover_vazamento)

if not suspeitas_correlacao.empty:
    print(f"\nAtencao: variavel(eis) com correlacao > {LIMITE_ALERTA_CORRELACAO} com o target (revisar manualmente):")
    print(suspeitas_correlacao)
else:
    print(f"\nNenhuma variavel numerica com correlacao > {LIMITE_ALERTA_CORRELACAO} encontrada.")


C:\Users\walla\AppData\Local\Temp\ipykernel_7028\2609806322.py:28: DtypeWarning: Columns (0: q10, 1: q12, 2: q14, 3: q14a, 4: q15, 5: q21a, 6: q22, 7: q23a, 8: q24, 9: q30, 10: q31, 11: q32a, 12: q33, 13: q35, 14: q37a, 15: q37b, 16: q38a, 17: q38b, 18: q41, 19: q43, 20: q43a, 21: q55a, 22: q57, 23: q58, 24: q59, 25: q63, 26: q73, 27: q76, 28: q77, 29: q78, 30: q85, 31: q86, 32: q87, 33: r101, 34: r102, 35: r103, 36: r104, 37: r105, 38: flvreg, 39: flvreco, 40: gordura, 41: leiteint, 42: refri5, 43: atilaz, 44: inativo, 45: regiao, 46: refritl5, 47: feijao5, 48: ati_l, 49: ativo_livre, 50: ina_livre, 51: inat_des_trab, 52: inat_des_esc, 53: inat_desl, 54: atilaz_trans, 55: q16, 56: q21, 57: q23, 58: q32, 59: q34, 60: q34a, 61: q34b, 62: q34c, 63: q34d, 64: q39, 65: q40, 66: q40a, 67: q72, 68: q79, 69: q80, 70: q81, 71: q82, 72: q83, 73: q83a, 74: r107, 75: r108, 76: iddmamo, 77: mamo, 78: mamodois, 79: papa, 80: papatres, 81: q25, 82: q26, 83: q66a, 84: r109, 85: r110, 86: r111, 87: r1

Atencao: 2 variavel(eis) do variaveis.txt NAO encontrada(s) na base:
- r701_time
- sono_curto

Base carregada. Linhas: 833217 | Colunas restantes: 91
Variaveis removidas por possivel vazamento (nome): ['ind_med_has', 'med_has', 'trat_med_has']

Atencao: variavel(eis) com correlacao > 0.3 com o target (revisar manualmente):
ind_med_db    0.373472
med_db        0.359937
iddpapa       0.337290
dtype: float64


## Bloco 3: Divisão de Dados de Treino e Teste e Estrutura do Pré-processador
Separação de dados em treino e teste mantendo a proporção original de doentes, além da construção do ColumnTransformer padrão para tratar missing e escala.

In [5]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category", "string"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
    ]
)

## Bloco 3.5: Duplicação das Bases de Treino/Teste e Criação de Novos Indicadores
Este bloco isola nosso laboratório de dados. Criamos cópias dos dados de treino e teste para construir novos scores e indicadores clínicos cruzados, garantindo que a base original do grupo (nosso Baseline) continue intacta e protegida contra alterações acidentais.

In [6]:
# Criação de cópias limpas para proteger a linha de base original (Baseline)
X_treino_custom = X_treino.copy()
X_teste_custom = X_teste.copy()

def aplicar_engenharia_features(df_target):
    df_out = df_target.copy()
    
    # 1. Risco Combinado Peso-Idade
    # O Vigitel às vezes muda o nome da coluna de idade dependendo do ano da base.
    # Essa checagem serve para o código não quebrar se a coluna se chamar 'iddpapa_old'.
    col_idade = 'iddpapa' if 'iddpapa' in df_out.columns else ('iddpapa_old' if 'iddpapa_old' in df_out.columns else None)
    if 'imc' in df_out.columns and col_idade:
        df_out['risco_peso_idade'] = pd.to_numeric(df_out['imc'], errors='coerce') * pd.to_numeric(df_out[col_idade], errors='coerce')
    
    # 2. Score de Síndrome Metabólica
    # Juntei todas as colunas de doenças crônicas que se relacionam com o coração
    fatores = ['diab', 'dislip', 'excpeso', 'excpeso_i', 'obesid']
    fatores_validos = [f for f in fatores if f in df_out.columns]
    if fatores_validos:
        df_out['score_sindrome_metabolica'] = df_out[fatores_validos].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
        
    # 3. Balanço Nutricional (Dieta Saudável vs Dieta Inflamatória)
    saudaveis = ['frutadia', 'frutareg', 'hortadia', 'hortareg', 'feijao5']
    riscos = ['gordura', 'doces5', 'refritl5', 'leiteint']
    s_validos = [col for col in saudaveis if col in df_out.columns]
    r_validos = [col for col in riscos if col in df_out.columns]
    
    nota_saudavel = df_out[s_validos].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1) if s_validos else 0
    nota_risco = df_out[r_validos].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1) if r_validos else 0
    # Se a nota der negativa, significa que a pessoa come pior do que come bem.
    df_out['balanco_nutricional'] = nota_saudavel - nota_risco
    
    # 4. Índice de Estresse Crônico e Distúrbio de Sono
    col_sono = 'insonia' if 'insonia' in df_out.columns else ('sono_curto' if 'sono_curto' in df_out.columns else None)
    if 'depressao' in df_out.columns and col_sono:
        df_out['indice_estresse_sono'] = pd.to_numeric(df_out['depressao'], errors='coerce').fillna(0) * pd.to_numeric(df_out[col_sono], errors='coerce').fillna(0)
        
    return df_out

# Aplica a função de engenharia nas duas bases de forma isolada
X_treino_custom = aplicar_engenharia_features(X_treino_custom)
X_teste_custom = aplicar_engenharia_features(X_teste_custom)

# ATENÇÃO AQUI: Como criamos colunas novas, precisamos rodar o seletor de tipos 
# de novo para o preprocessor saber que essas colunas novas numéricas existem!
num_cols_custom = X_treino_custom.select_dtypes(include=["int64", "float64"]).columns
cat_cols_custom = X_treino_custom.select_dtypes(include=["object", "category", "string"]).columns

preprocessor_custom = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols_custom),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols_custom)
    ]
)

print(f"Sucesso! Novas variáveis clínicas acopladas.")
print(f"Colunas originais: {X_treino.shape[1]} -> Novas colunas customizadas: {X_treino_custom.shape[1]}")

Sucesso! Novas variáveis clínicas acopladas.
Colunas originais: 91 -> Novas colunas customizadas: 95


## Bloco 4: Execução do Pipeline Original (Competição de Modelos e Geração do Baseline)
O motor de busca testa de forma combinada a Regressão Logística, Árvore de Decisão e Random Forest usando os parâmetros definidos pela equipe e salvando os gráficos na pasta.

In [7]:
# ============================================================
# MODELOS-BASE E GRADES DE HIPERPARÂMETROS PARA TUNING
# ============================================================
# Definimos os dicionários contendo os modelos e as listas de botões
# de regulagem que o GridSearchCV vai testar de forma combinada.

modelos_base = {
    "Regressao_Logistica": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random_Forest": RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=1,
        class_weight="balanced"
    ),
    "Arvore_Decisao": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

grades_hiperparametros = {
    "Regressao_Logistica": {
        "modelo__C": [0.01, 0.1, 1, 10]
    },
    "Random_Forest": {
        "modelo__n_estimators": [200, 500],
        "modelo__max_depth": [8, 12],
        "modelo__min_samples_leaf": [5, 10]
    },
    "Arvore_Decisao": {
        "modelo__max_depth": [6, 10, 14],
        "modelo__min_samples_leaf": [5, 10, 20]
    }
}

# Configura as divisões para as rodadas de validação cruzada
cv_tuning = StratifiedKFold(n_splits=N_FOLDS_TUNING, shuffle=True, random_state=RANDOM_STATE)
cv_avaliacao = StratifiedKFold(n_splits=N_FOLDS_AVALIACAO, shuffle=True, random_state=RANDOM_STATE)

metricas_cv = ["accuracy", "precision", "recall", "f1", "roc_auc"]

# Listas auxiliares para armazenar os relatórios
resultados = []
resultados_cv = []
resultados_cv_folds = {}
melhores_parametros = {}
modelos_treinados = {}
curvas_roc = {}
curvas_pr = {}

# ============================================================
# EXECUÇÃO DO LAÇO DE TREINAMENTO E VALIDAÇÃO
# ============================================================
for nome_modelo, algoritmo_base in modelos_base.items():
    print("\n" + "=" * 80)
    print(f"Processando o Modelo: {nome_modelo}")
    print("=" * 80)

    # Conecta a esteira automática ao algoritmo atual da rodada
    pipeline_base = Pipeline([
        ("prep", preprocessor),
        ("modelo", algoritmo_base)
    ])

    # 1) Executa o Tuning de Hiperparâmetros no conjunto de treino
    print(f"Buscando melhores hiperparâmetros ({N_FOLDS_TUNING} folds)...")
    grid = GridSearchCV(
        estimator=pipeline_base,
        param_grid=grades_hiperparametros[nome_modelo],
        scoring="roc_auc",
        cv=cv_tuning,
        n_jobs=N_JOBS_CONFIGURADO,  # ajuste N_JOBS_CONFIGURADO no Bloco 2 conforme sua máquina
        refit=True
    )
    grid.fit(X_treino, y_treino)

    pipeline = grid.best_estimator_
    melhores_parametros[nome_modelo] = grid.best_params_

    print(f"Melhores parâmetros encontrados: {grid.best_params_}")
    print(f"Melhor AUC-ROC durante o tuning: {grid.best_score_:.4f}")

    # 2) Executa a Validação Cruzada de 5 Folds para conferência estável
    print(f"Executando validação cruzada de avaliação ({N_FOLDS_AVALIACAO} folds)...")
    cv_resultado = cross_validate(
        pipeline, X, y, cv=cv_avaliacao, scoring=metricas_cv, n_jobs=N_JOBS_CONFIGURADO
    )

    linha_cv = {"modelo": nome_modelo}
    for metrica in metricas_cv:
        chave = f"test_{metrica}"
        linha_cv[f"{metrica}_media"] = cv_resultado[chave].mean()
        linha_cv[f"{metrica}_desvio"] = cv_resultado[chave].std()

    resultados_cv.append(linha_cv)
    resultados_cv_folds[nome_modelo] = cv_resultado

    # 3) Faz a avaliação final usando o split isolado de Teste
    pred = pipeline.predict(X_teste)
    proba = pipeline.predict_proba(X_teste)[:, 1]

    acc = accuracy_score(y_teste, pred)
    precision = precision_score(y_teste, pred, zero_division=0)
    recall = recall_score(y_teste, pred, zero_division=0)
    f1 = f1_score(y_teste, pred, zero_division=0)
    auc = roc_auc_score(y_teste, proba)

    resultados.append({
        "modelo": nome_modelo, "accuracy": acc, "precision": precision,
        "recall": recall, "f1_score": f1, "roc_auc": auc
    })

    modelos_treinados[nome_modelo] = pipeline

    print("\nClassification Report (split treino/teste):")
    print(classification_report(y_teste, pred, zero_division=0))

    # Coleta as taxas para plotar as curvas comparativas mais tarde
    fpr, tpr, _ = roc_curve(y_teste, proba)
    curvas_roc[nome_modelo] = (fpr, tpr, auc)
    precisao_pr, recall_pr, _ = precision_recall_curve(y_teste, proba)
    ap = average_precision_score(y_teste, proba)
    curvas_pr[nome_modelo] = (precisao_pr, recall_pr, ap)

    # 4) Desenha e salva a Matriz de Confusão com termos amigáveis
    ConfusionMatrixDisplay.from_predictions(
        y_teste, pred, display_labels=["Sem Hipertensão", "Com Hipertensão"],
        values_format="d", cmap="Blues"
    )
    plt.title(f"Matriz de Confusão - {nome_modelo}")
    plt.tight_layout()
    caminho_matriz = os.path.join(PASTA_RESULTADOS, f"matriz_confusao_{nome_modelo}.png")
    plt.savefig(caminho_matriz, dpi=300)
    plt.close()

    # 5) Exporta o arquivo físico do modelo pronto para reuso (.joblib)
    caminho_modelo_salvo = os.path.join(PASTA_MODELOS, f"{nome_modelo}.joblib")
    joblib.dump(pipeline, caminho_modelo_salvo)

# ============================================================
# EXPORTAÇÃO DE RELATÓRIOS EM TEXTO E CSV
# ============================================================
caminho_hiperparametros = os.path.join(PASTA_RESULTADOS, "melhores_hiperparametros.txt")
with open(caminho_hiperparametros, "w", encoding="utf-8") as arquivo:
    for nome_modelo, params in melhores_parametros.items():
        arquivo.write(f"{nome_modelo}: {params}\n")

resultados_cv_df = pd.DataFrame(resultados_cv).sort_values(by="roc_auc_media", ascending=False)
resultados_cv_df.to_csv(os.path.join(PASTA_RESULTADOS, "comparacao_modelos_cross_validation_sem_vazamento.csv"), index=False, encoding="utf-8-sig")

resultados_df = pd.DataFrame(resultados).sort_values(by="roc_auc", ascending=False)
resultados_df.to_csv(os.path.join(PASTA_RESULTADOS, "comparacao_modelos_sem_vazamento.csv"), index=False, encoding="utf-8-sig")

# ============================================================
# DESENHO DOS GRÁFICOS VISUAIS PARA O PDF FINAL
# ============================================================
# Gráfico de Linhas: Comparativo de Métricas Gerais
plt.figure(figsize=(10, 6))
x_axis = resultados_df["modelo"]
plt.plot(x_axis, resultados_df["accuracy"], marker="o", label="Accuracy")
plt.plot(x_axis, resultados_df["precision"], marker="o", label="Precision")
plt.plot(x_axis, resultados_df["recall"], marker="o", label="Recall")
plt.plot(x_axis, resultados_df["f1_score"], marker="o", label="F1-score")
plt.plot(x_axis, resultados_df["roc_auc"], marker="o", label="AUC-ROC")
plt.title("Comparação de Métricas dos Modelos (split treino/teste)")
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_RESULTADOS, "comparacao_metricas_modelos_sem_vazamento.png"), dpi=300)
plt.close()

# Gráfico de Linhas: Curva ROC Comparativa
plt.figure(figsize=(8, 8))
for nome_modelo, (fpr, tpr, auc_modelo) in curvas_roc.items():
    plt.plot(fpr, tpr, label=f"{nome_modelo} (AUC = {auc_modelo:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Modelo aleatório")
plt.xlabel("Taxa de Falsos Positivos")
plt.ylabel("Taxa de Verdadeiros Positivos (Recall)")
plt.title("Curva ROC - Comparação entre Modelos")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_RESULTADOS, "curva_roc_comparativa.png"), dpi=300)
plt.close()

# Extração de Feature Importance (Random Forest)
modelo_rf = modelos_treinados["Random_Forest"]
feature_names = modelo_rf.named_steps["prep"].get_feature_names_out()
importance = modelo_rf.named_steps["modelo"].feature_importances_

importance_df = pd.DataFrame({"variavel": feature_names, "importancia": importance}).sort_values("importancia", ascending=False)
importance_df.to_csv(os.path.join(PASTA_RESULTADOS, "feature_importance_random_forest_sem_vazamento.csv"), index=False, encoding="utf-8-sig")

# Gráfico de Barras: Top 20 variáveis explicativas do modelo campeão
top20 = importance_df.head(20)
plt.figure(figsize=(10, 8))
plt.barh(top20["variavel"], top20["importancia"])
plt.gca().invert_yaxis()
plt.title("Top 20 Variáveis Mais Importantes - Random Forest")
plt.tight_layout()
plt.savefig(os.path.join(PASTA_RESULTADOS, "top20_feature_importance_random_forest_sem_vazamento.png"), dpi=300)
plt.close()

# Imprime o grande veredito na tela do Jupyter
print("\n" + "=" * 80)
print("DIAGNÓSTICO DO PIPELINE CONCLUÍDO COM SUCESSO")
print("=" * 80)
melhor_modelo_cv = resultados_cv_df.iloc[0]
print(f"Modelo Vencedor (Validação Cruzada): {melhor_modelo_cv['modelo']}")
print(f"AUC-ROC Médio obtido: {melhor_modelo_cv['roc_auc_media']:.4f} ± {melhor_modelo_cv['roc_auc_desvio']:.4f}")
print(f"Todos os artefatos visuais e relatórios foram salvos na pasta: {PASTA_RESULTADOS}")


Processando o Modelo: Regressao_Logistica
Buscando melhores hiperparâmetros (3 folds)...
Melhores parâmetros encontrados: {'modelo__C': 10}
Melhor AUC-ROC durante o tuning: 0.7728
Executando validação cruzada de avaliação (5 folds)...

Classification Report (split treino/teste):
              precision    recall  f1-score   support

           0       0.84      0.72      0.77    173408
           1       0.52      0.69      0.59     76558

    accuracy                           0.71    249966
   macro avg       0.68      0.70      0.68    249966
weighted avg       0.74      0.71      0.72    249966


Processando o Modelo: Random_Forest
Buscando melhores hiperparâmetros (3 folds)...
Melhores parâmetros encontrados: {'modelo__max_depth': 12, 'modelo__min_samples_leaf': 5, 'modelo__n_estimators': 500}
Melhor AUC-ROC durante o tuning: 0.7734
Executando validação cruzada de avaliação (5 folds)...

Classification Report (split treino/teste):
              precision    recall  f1-score   sup

## Bloco 4.5: O Experimento Customizado (Otimização de Precisão)
Cenário focado na redução radical de alarmes falsos para os médicos. Treinamos a Random Forest sem balanceamento artificial de classes e alteramos o gatilho de decisão de 50% para 60%, forçando o modelo a só dar diagnóstico se tiver alto grau de certeza matemática.

In [8]:
# ============================================================
# BLOCO 4.5 (INVERTIDO) - OTIMIZAÇÃO DE RECALL / SENSIBILIDADE
# ============================================================
# Objetivo invertido: priorizar NÃO deixar doente passar (menos FN), mesmo
# que isso aumente os falsos alarmes (mais FP). Duas mudanças em relação à
# versão original:
#   1) GridSearchCV agora otimiza para "recall" em vez de "precision".
#   2) O limiar de decisão é REDUZIDO (em vez de elevado), pra classificar
#      como positivo com menos certeza exigida do modelo.

print("\n" + "=" * 80)
print("INICIANDO EXPERIMENTO CUSTOMIZADO PARA MAXIMIZAR RECALL CLÍNICO (SENSIBILIDADE)")
print("=" * 80)

cv_tuning = StratifiedKFold(n_splits=N_FOLDS_TUNING, shuffle=True, random_state=RANDOM_STATE)

# DETALHE IMPORTANTE: mantemos/adicionamos class_weight="balanced" para o
# modelo prestar mais atenção na classe minoritária (quem tem hipertensão),
# em vez de "relaxar" como fazia a versão focada em precisão.
pipeline_custom = Pipeline([
    ("prep", preprocessor_custom),
    ("modelo", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1, class_weight="balanced"))
])

grade_custom = {
    "modelo__n_estimators": [200],
    "modelo__max_depth": [12, 16],
    "modelo__min_samples_leaf": [5]
}

# MUDANÇA DE DIREÇÃO: scoring="recall" em vez de "precision" - o tuning
# agora escolhe os hiperparâmetros que capturam MAIS casos positivos.
grid_custom = GridSearchCV(
    estimator=pipeline_custom,
    param_grid=grade_custom,
    scoring="recall",
    cv=cv_tuning,
    n_jobs=N_JOBS_CONFIGURADO,
    refit=True
)
grid_custom.fit(X_treino_custom, y_treino)

best_pipeline_custom = grid_custom.best_estimator_

probabilidades_teste = best_pipeline_custom.predict_proba(X_teste_custom)[:, 1]

# MUDANÇA DE DIREÇÃO: limiar REDUZIDO (era 0.60, agora 0.35). Em vez de
# chutar esse número, o ideal é escolhê-lo olhando o gráfico de FP x FN
# gerado mais abaixo - 0.35 é só um ponto de partida razoável.
LIMIAR_OTIMIZADO = 0.35
predicoes_otimizadas = (probabilidades_teste >= LIMIAR_OTIMIZADO).astype(int)

print(f"\nMelhores parâmetros encontrados no Tuning Customizado: {grid_custom.best_params_}")
print(f"\nClassification Report - Modelo Otimizado (Novas Features + Threshold {int(LIMIAR_OTIMIZADO*100)}%):")
print(classification_report(y_teste, predicoes_otimizadas, zero_division=0))

precisao_otimizada = precision_score(y_teste, predicoes_otimizadas, zero_division=0)
recall_otimizado = recall_score(y_teste, predicoes_otimizadas, zero_division=0)
print(f"Recall do cenário otimizado (deve subir bastante): {recall_otimizado:.4f}")
print(f"Precisão do cenário otimizado (cai por design - mais alarmes falsos): {precisao_otimizada:.4f}")

ConfusionMatrixDisplay.from_predictions(
    y_teste,
    predicoes_otimizadas,
    display_labels=["Sem Hipertensão", "Com Hipertensão"],
    values_format="d",
    cmap="Oranges"
)
plt.title("Matriz de Confusão Otimizada (Foco em Recall/Sensibilidade)")
plt.tight_layout()
plt.savefig(os.path.join(PASTA_RESULTADOS, "matriz_confusao_RandomForest_Otimizada_Recall.png"), dpi=300)
plt.close()

joblib.dump(best_pipeline_custom, os.path.join(PASTA_MODELOS, "Random_Forest_Otimizada_Recall.joblib"))
print("\nModelo otimizado para recall exportado com sucesso na pasta de execuções atuais.")


# ============================================================
# GRÁFICO - TRADE-OFF ENTRE FALSOS POSITIVOS E FALSOS NEGATIVOS
# ============================================================
# Varia o limiar de decisão de 1% a 99% e, para cada um, calcula quantos
# Falsos Positivos (FP) e Falsos Negativos (FN) o modelo gera. Isso permite
# ESCOLHER um limiar com base em dados, em vez de chutar um número.

import numpy as np
from sklearn.metrics import confusion_matrix

limiares = np.linspace(0.01, 0.99, 99)
falsos_positivos = []
falsos_negativos = []

for limiar in limiares:
    predicoes = (probabilidades_teste >= limiar).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_teste, predicoes).ravel()
    falsos_positivos.append(fp)
    falsos_negativos.append(fn)

plt.figure(figsize=(8, 8))
dispersao = plt.scatter(falsos_positivos, falsos_negativos, c=limiares, cmap="viridis", s=20)
plt.plot(falsos_positivos, falsos_negativos, alpha=0.3, color="gray")
plt.colorbar(dispersao, label="Limiar de decisão")
plt.scatter(
    [falsos_positivos[np.argmin(np.abs(limiares - LIMIAR_OTIMIZADO))]],
    [falsos_negativos[np.argmin(np.abs(limiares - LIMIAR_OTIMIZADO))]],
    color="red", marker="*", s=300, label=f"Limiar escolhido ({LIMIAR_OTIMIZADO})", zorder=5
)
plt.xlabel("Falsos Positivos (FP)")
plt.ylabel("Falsos Negativos (FN)")
plt.title("Trade-off entre Falsos Positivos e Falsos Negativos por Limiar")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

caminho_grafico_fp_fn = os.path.join(PASTA_RESULTADOS, "tradeoff_fp_fn_por_limiar.png")
plt.savefig(caminho_grafico_fp_fn, dpi=300)
plt.close()

print(f"Gráfico de trade-off FP x FN salvo em: {caminho_grafico_fp_fn}")


INICIANDO EXPERIMENTO CUSTOMIZADO PARA MAXIMIZAR RECALL CLÍNICO (SENSIBILIDADE)

Melhores parâmetros encontrados no Tuning Customizado: {'modelo__max_depth': 16, 'modelo__min_samples_leaf': 5, 'modelo__n_estimators': 200}

Classification Report - Modelo Otimizado (Novas Features + Threshold 35%):
              precision    recall  f1-score   support

           0       0.90      0.46      0.61    173408
           1       0.42      0.89      0.57     76558

    accuracy                           0.59    249966
   macro avg       0.66      0.67      0.59    249966
weighted avg       0.75      0.59      0.60    249966

Recall do cenário otimizado (deve subir bastante): 0.8860
Precisão do cenário otimizado (cai por design - mais alarmes falsos): 0.4213

Modelo otimizado para recall exportado com sucesso na pasta de execuções atuais.
Gráfico de trade-off FP x FN salvo em: C:\Users\walla\OneDrive\Desktop\Fiap\Fase 1\pos-tech-ia-tech-challenge-fase1\resultados_sem_vazamento\20260623_212446\

## Bloco 4.6: O Experimento Customizado (Apenas Novas Variáveis)
Cenário científico de isolamento de impacto. Mantemos todas as regras originais do grupo (corte padrão em 50% e peso de classe balanceado) para medir de forma justa o ganho incremental que as nossas variáveis trouxeram sozinhas para a Random Forest.

In [9]:
print("\n" + "=" * 80)
print("TESTANDO APENAS O IMPACTO DAS NOVAS VARIÁVEIS (THRESHOLD 50%)")
print("=" * 80)

cv_tuning = StratifiedKFold(n_splits=N_FOLDS_TUNING, shuffle=True, random_state=RANDOM_STATE)

# 1. Adicionado o class_weight="balanced" 
pipeline_custom = Pipeline([
    ("prep", preprocessor_custom),
    ("modelo", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1, class_weight="balanced")) # n_jobs=1 evita paralelismo aninhado
])

grade_custom = {
    "modelo__n_estimators": [200],
    "modelo__max_depth": [12, 16],
    "modelo__min_samples_leaf": [5]
}

# CORRECAO: usamos "roc_auc" (igual ao Bloco 4/baseline) em vez de
# "precision", para que a unica diferenca entre este cenario e o baseline
# sejam as novas variaveis - e nao tambem o criterio de tuning.
grid_custom = GridSearchCV(estimator=pipeline_custom, param_grid=grade_custom, scoring="roc_auc", cv=cv_tuning, n_jobs=N_JOBS_CONFIGURADO, refit=True)
grid_custom.fit(X_treino_custom, y_treino)

best_pipeline_custom = grid_custom.best_estimator_

# 2. Limiar de 50%
probabilidades_teste = best_pipeline_custom.predict_proba(X_teste_custom)[:, 1]
LIMIAR_OTIMIZADO = 0.50 
predicoes_otimizadas = (probabilidades_teste >= LIMIAR_OTIMIZADO).astype(int)

print(f"\nMelhores parâmetros encontrados: {grid_custom.best_params_}")
print("\nClassification Report - Teste Isolado de Novas Variáveis (Threshold 50%):")
print(classification_report(y_teste, predicoes_otimizadas, zero_division=0))

# Salvamento da Matriz de Confusão do Experimento
ConfusionMatrixDisplay.from_predictions(y_teste, predicoes_otimizadas, display_labels=["Sem Hipertensão", "Com Hipertensão"], values_format="d", cmap="Greens")
plt.title("Matriz de Confusão - Apenas Novas Variáveis (Limiar 50%)")
plt.tight_layout()
plt.savefig(os.path.join(PASTA_RESULTADOS, "matriz_confusao_RandomForest_Somente_Variaveis.png"), dpi=300)
plt.close()


TESTANDO APENAS O IMPACTO DAS NOVAS VARIÁVEIS (THRESHOLD 50%)

Melhores parâmetros encontrados: {'modelo__max_depth': 16, 'modelo__min_samples_leaf': 5, 'modelo__n_estimators': 200}

Classification Report - Teste Isolado de Novas Variáveis (Threshold 50%):
              precision    recall  f1-score   support

           0       0.85      0.70      0.77    173408
           1       0.51      0.71      0.60     76558

    accuracy                           0.70    249966
   macro avg       0.68      0.71      0.68    249966
weighted avg       0.74      0.70      0.71    249966

